In [153]:
import urllib
from tqdm import tqdm
import requests
import pandas as pd
import re
import json
import datetime
from github_helper import from_github

## This notebook adds which politician it is, and what their party is


In [154]:
df = pd.read_csv(from_github("/voting-data/df_votes_all_periods.csv"))

In [163]:
request_session = requests.Session()

def get_actor_relations(aktør_id, session = request_session):
    # all_relations = []
    base_url = "https://oda.ft.dk/api/AktørAktør"
    params = {
        '$filter': 
            f'fraaktørid eq {aktør_id} and rolleid eq 15'
        ,'$expand': 'TilAktør, FraAktør'
    }
    response = session.get(base_url, params=params)
    # print(response.url)
    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {aktør_id}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
    relations = data.get('value')

    return relations

def build_parties_from_relations(relations_for_actor):
    all_relations = []
    for relation in relations_for_actor:
        kendt_aktør = relation.get('FraAktør')
        kendt_aktør_navn = kendt_aktør.get('navn')
        kendt_aktør_id = kendt_aktør.get('id')
        anden_aktør = relation.get('TilAktør')
        relation_start = relation.get('startdato')
        typeid = anden_aktør.get('typeid')
        if typeid == 4: #Typeid 4 means it's a party
            party_name = anden_aktør.get('navn')
            info = [kendt_aktør_id, kendt_aktør_navn, party_name, relation_start]
            all_relations.append(info)
    
    relations_df = pd.DataFrame(all_relations, columns= ["aktørid", "aktør", "party", "relation_start"])
    
    return relations_df


def normalize_party(name: str) -> str:
    """Map raw party string to canonical party name."""
    if pd.isna(name):
        return name

    for pattern, canonical in party_naming_rules:
        if re.match(pattern, name):
            return canonical

    # If nothing matches, just return original
    return name

party_naming_rules = [
    # Big Danish parties
    (r'^Enhedslisten',                              'Enhedslisten'),
    (r'^Socialdemokratiet$',                        'Socialdemokratiet'),
    (r'^Socialistisk Folkeparti$',                  'Socialistisk Folkeparti'),
    (r'^Dansk Folkeparti$',                         'Dansk Folkeparti'),
    (r'^Venstre, Danmarks Liberale Parti$',         'Venstre'),
    (r'^Det Radikale Venstre$',                     'Radikale Venstre'),
    (r'^Radikale Venstre$',                         'Radikale Venstre'),
    (r'^Det Konservative Folkeparti$',              'Det Konservative Folkeparti'),
    (r'^Liberal Alliance$',                         'Liberal Alliance'),
    (r'^Moderaterne$',                              'Moderaterne'),
    (r'^Ny Alliance$',                              'Liberal Alliance'),
    (r'^Alternativet$',                             'Alternativet'),
    (r'^Frie Grønne, Danmarks Nye Venstrefløjsparti$', 'Frie Grønne'),
    (r'^Venstresocialisterne$',                     'Venstresocialisterne'),

    # "Uden for folketingsgrupperne - <navn>"
    (r'^Uden for folketingsgrupperne\b',            'Uden for Folketingsgrupperne'),

    # Danmarksdemokraterne variants (dash vs en dash)
    (r'^Danmarksdemokraterne\b',                    'Danmarksdemokraterne'),

    # Kristendemokraterne / Kristeligt Folkeparti (same party, renamed)
    # If you prefer to keep them separate, split these into two different canonicals.
    (r'^Kristendemokraterne$',                      'Kristendemokraterne'),
    (r'^Kristeligt Folkeparti$',                    'Kristendemokraterne'),

    # Other Danish / Danish-rooted parties
    (r'^Fremskridtspartiet$',                       'Fremskridtspartiet'),
    (r'^Frihed 2000$',                              'Frihed 2000'),
    (r'^Borgernes Parti\b',                         'Borgernes Parti'),
    (r'^Nye Borgerlige$',                           'Nye Borgerlige'),

    # Greenlandic parties
    (r'^Inuit Ataqatigiit$',                        'Inuit Ataqatigiit'),
    (r'^Siumut$',                                   'Siumut'),
    (r'^Nunatta Qitornai$',                         'Nunatta Qitornai'),
    (r'^Naleraq$',                                  'Naleraq'),

    # Faroese parties
    (r'^Sambandsflokkurin$',                        'Sambandsflokkurin'),
    (r'^Javnaðarflokkurin$',                        'Javnaðarflokkurin'),
    (r'^Tjóðveldisflokkurin$',                      'Tjóðveldi'),
    (r'^Tjóðveldi$',                                'Tjóðveldi'),
]


def make_party_intervals_from_all_actors(relations_df_for_all):
    # where_has_relation_start = (~relations_df_for_all['relation_start'].isnull())
    df_rel = relations_df_for_all.copy()
    df_rel['relation_start'] = pd.to_datetime(df_rel['relation_start'])

    # True if this aktørid has at least one non-null relation_start
    has_any_start = df_rel.groupby('aktørid')['relation_start'].transform('count') > 0

    # Rows belonging to aktørid where *all* relation_start are NaT
    mask_no_start_actor = ~has_any_start

    # First row per aktørid (based on current ordering)
    first_row_per_actor = ~df_rel.duplicated(subset='aktørid', keep='first')

    # Default start date
    default_start = pd.Timestamp('2000-01-01T12').normalize()

    # For actors with no start dates at all, set first row's relation_start
    df_rel.loc[mask_no_start_actor & first_row_per_actor, 'relation_start'] = default_start

    # where_has_relation_start_mask = (~relations_df_for_all['relation_start'].isnull())
    where_has_relation_start = df_rel['relation_start'].notna()
    df_rel = df_rel[where_has_relation_start].copy()

    # Sort so "next" makes sense (per actor)
    df_rel = df_rel.sort_values(['aktørid', 'relation_start'])

    # Next relation_start per actor
    df_rel['relation_end'] = (
        df_rel
        .groupby('aktørid')['relation_start']
        .shift(-1)
        - pd.Timedelta(days=1)
    )

    # 1) For each aktørid, detect when the party changes vs previous row
    party_change = (
        df_rel['party']
        != df_rel.groupby('aktørid')['party'].shift()
    )

    # 2) Use cumulative sum to create a "block" id of consecutive same-party rows
    df_rel['block'] = party_change.groupby(df_rel['aktørid']).cumsum()

    # 3) Group by aktørid + party + block and aggregate start/end
    collapsed = (
        df_rel
        .groupby(['aktørid', 'aktør', 'party', 'block'], as_index=False)
        .agg(
            relation_start=('relation_start', 'min'),
            relation_end=('relation_end', 'max')
        )
    )
    where_no_end = (collapsed['relation_end'].isnull())
    collapsed.loc[where_no_end, 'relation_end'] = pd.Timestamp.today().normalize()
    collapsed.drop(columns = "block", inplace = True)
    collapsed = collapsed.sort_values(['aktørid', 'relation_start'])

    collapsed['party_clean'] = collapsed['party'].apply(normalize_party)
    old_parties = collapsed['party'].unique()
    new_parties = collapsed['party_clean'].unique()
    print(f"{len(old_parties)-len(new_parties)} 'parties' removed in cleaning, where some are like 'Uden for folketingsgrupperne - <SomeName>'. The mapping can be found in 'party_naming_rules'.")
    collapsed.drop(columns = "party", inplace=True)
    collapsed.rename(columns = {"party_clean" : "party"}, inplace=True)
    
    return collapsed



       
    
################################## TESTING, NOT TO BE INCLUDED IN ANY FINAL CODE
#12 #Nicolai Vammen
# aktør_id = 18723 #Vermund
# aktør_id = 77 #Kristian Thulesen Dahl
# aktør_id = 8319 #WHO IS THIS? I guess we will never know
# aktør_id = 1440
# aktør_id = 3089
# aktør_id = 3083
aktør_id = 220

relations_for_actor = get_actor_relations(aktør_id)
relations_df = build_parties_from_relations(relations_for_actor=relations_for_actor)
relations_for_actor
relations_df
party_int = make_party_intervals_from_all_actors(relations_df)
party_int


0 'parties' removed in cleaning, where some are like 'Uden for folketingsgrupperne - <SomeName>'. The mapping can be found in 'party_naming_rules'.


,aktørid,aktør,relation_start,relation_end,party
0,220,Morten Bødskov,2002-10-01,2022-10-03,Socialdemokratiet


In [156]:
def get_actor_df_with_no_relations(aktør_id, session = request_session):
    url = f"https://oda.ft.dk/api/Aktør({aktør_id})"
    response = session.get(url)

    # print(response.url)
    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {aktør_id}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
    
    kendt_aktør_id = aktør_id
    kendt_aktør_navn = data.get('navn')
    party_name = "N/A"
    relation_start = pd.Timestamp('2000-01-01T12').normalize()
    # relation_start = "2000-01-01T00:00:00"
    dict_info = {"aktørid" : [kendt_aktør_id], "aktør" : [kendt_aktør_navn], "party" : [party_name] , "relation_start" : [relation_start]}
    relations_df = pd.DataFrame(data = dict_info)

    return relations_df, kendt_aktør_navn, kendt_aktør_id
    

# aktørid = 18723#12
# aktørid = 8319

In [164]:
def generate_party_per_period_per_actor(party_intervals):
    from folketingsperioder import periods_df

    # ----- 1) Actor × Period Cartesian product -----
    actors = party_intervals[['aktørid', 'aktør']].drop_duplicates()
    periods_subset = periods_df[
        periods_df['Period'].between(65, 71)
    ][["Period", "period_start_date"]].copy()

    actors['key'] = 1
    periods_subset['key'] = 1
    actor_periods = actors.merge(periods_subset, on='key').drop(columns='key')

    # Clean inputs
    ap = actor_periods.dropna(subset=['aktørid', 'period_start_date']).copy()
    intervals = party_intervals.dropna(subset=['aktørid', 'relation_start']).copy()

    # Sort globally on the merge key (IMPORTANT for merge_asof)
    ap_sorted = ap.sort_values('period_start_date').reset_index(drop=True)
    intervals_sorted = intervals.sort_values('relation_start').reset_index(drop=True)
    intervals_sorted = intervals_sorted.drop(columns=['aktør'])

    # ----- 2) BACKWARD merge: party before period start -----
    backward = pd.merge_asof(
        ap_sorted,
        intervals_sorted,
        left_on='period_start_date',
        right_on='relation_start',
        by='aktørid',
        direction='backward',
        allow_exact_matches=True
    )
    backward.rename(columns={'party': 'party_back'}, inplace=True)

    # ----- 3) FORWARD merge: party after period start (mid-period joiners) -----
    forward = pd.merge_asof(
        ap_sorted,
        intervals_sorted,
        left_on='period_start_date',
        right_on='relation_start',
        by='aktørid',
        direction='forward',   # relation_start >= period_start_date
        allow_exact_matches=True
    )
    forward.rename(columns={'party': 'party_fwd'}, inplace=True)

    # Combine backward + forward
    merged = backward.copy()
    merged['party_fwd'] = forward['party_fwd']

    # ----- 4) Final party logic -----
    # 1) If there was a party BEFORE the period → use that
    # 2) Otherwise if party starts INSIDE the period → use that
    merged['party'] = merged['party_back'].fillna(merged['party_fwd'])

    # If still NA → actor has no party in that period → drop
    merged = merged.dropna(subset=['party'])

    result = merged[['aktørid', 'aktør', 'Period', 'party']].reset_index(drop=True)
    return result


In [165]:
unique_actors = df['aktørid'].unique()
all_relations_df = pd.DataFrame()
for aktørid in tqdm(unique_actors):
    relations_json = get_actor_relations(aktørid)
    if relations_json:
        relations_df = build_parties_from_relations(relations_json)
    else:
        relations_df, politician_name, aktør_id = get_actor_df_with_no_relations(aktørid)
        print(f"Found no party relations in AktørAktør for {politician_name} with aktørid {aktør_id}, created it from Aktør API instead")

    all_relations_df = pd.concat([all_relations_df, relations_df])

print(f"Created party relations for {len(all_relations_df['aktørid'].unique())} unique actors out of {df['aktørid'].nunique()} unique in df_votes.")

#There are some politicians which are not correctly mapped in the system. We manually map them.
map_actor_to_party ={
    20976: "Moderaterne" #Caroline Stage Olsen. We can assign this because, as we see later, we work with period 66 and 71. Caroline was not active in 66, and is in moderaterne in 71
    ,5905: "Socialdemokratiet" #Mogens Jensen
    ,3042: "Socialdemokratiet" #Frode Sørensen
    ,7633: "Socialdemokratiet" #Torben Hansen 
    ,5593: "Socialdemokratiet" #Carsten Hansen
    ,8319: "Socialdemokratiet" #Jytte Andersen
    }

for aktørid, party in map_actor_to_party.items():
    actor_observation_mask = (all_relations_df['aktørid']==aktørid)
    all_relations_df.loc[actor_observation_mask, "party"] = party

#now create party intervals for all politicians
party_intervals = make_party_intervals_from_all_actors(all_relations_df)
#based on the party interval, assign them to the party they were in at the start of the legislative period, for that period, so if they changed party, we do not consider it
party_per_period_df = generate_party_per_period_per_actor(party_intervals)

# party_intervals
party_per_period_df.to_csv("./actor-data/party_per_period.csv", index= False)

  4%|▍         | 31/710 [00:03<01:35,  7.10it/s]

Found no party relations in AktørAktør for Jytte Andersen with aktørid 8319, created it from Aktør API instead


 12%|█▏        | 83/710 [00:06<00:39, 15.68it/s]

Found no party relations in AktørAktør for Carsten Hansen with aktørid 5593, created it from Aktør API instead


 12%|█▏        | 88/710 [00:07<00:38, 16.28it/s]

...

Found no party relations in AktørAktør for Mogens Jensen, Brøndby with aktørid 5905, created it from Aktør API instead


 99%|█████████▉| 703/710 [00:50<00:00, 24.72it/s]

Found no party relations in AktørAktør for Caroline Stage Olsen with aktørid 20976, created it from Aktør API instead


100%|██████████| 710/710 [00:50<00:00, 13.93it/s]

Created party relations for 710 unique actors out of 710 unique in df_votes.
48 'parties' removed in cleaning, where some are like 'Uden for folketingsgrupperne - <SomeName>'. The mapping can be found in 'party_naming_rules'.
